In [ ]:
#@title **Dataset Captioner Studio** 🎒 ✨ { display-mode: "form" }
#@markdown This cell automatically sets up the environment, clones the repository, installs dependencies,
#@markdown downloads optimized llama-server binaries, starts the captioner, and tunnels it through Cloudflare.

import os
import re
import sys
import time
import shutil
import subprocess
import threading
from IPython.display import HTML, display, clear_output

# Create logs directory
os.makedirs("/content/studio_logs", exist_ok=True)
setup_log = "/content/studio_logs/setup.log"
captioner_log = "/content/studio_logs/captioner.log"
cf_log = "/content/studio_logs/cf.log"

# Initialize files
for fpath in [setup_log, captioner_log, cf_log]:
    with open(fpath, "w", encoding="utf-8") as f:
        f.write("")

steps = {
    "env": "Waiting",
    "git": "Waiting",
    "deps": "Waiting",
    "cf_install": "Waiting",
    "server": "Waiting",
    "tunnel": "Waiting"
}
tunnel_url = None
log_lines = []

def run_cmd(cmd, step_name):
    global steps, log_lines
    steps[step_name] = "Running"
    with open(setup_log, "a", encoding="utf-8") as f:
        f.write(f"--- Starting: {cmd} ---\n")
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in iter(proc.stdout.readline, ''):
        with open(setup_log, "a", encoding="utf-8") as f:
            f.write(line)
        log_lines.append(f"[Setup] {line.strip()}")
        if len(log_lines) > 20:
            log_lines.pop(0)
    proc.stdout.close()
    ret = proc.wait()
    if ret == 0:
        steps[step_name] = "Completed"
        return True
    else:
        steps[step_name] = "Failed"
        return False

def bg_setup():
    global tunnel_url, steps
    
    # 1. Env
    run_cmd("apt-get update -y -q && apt-get install -y -q aria2", "env")
    
    # 2. Git Clone
    if os.path.exists("Captioner-for-Colab"):
        log_lines.append("[Setup] Repository directory already exists. Pulling latest...")
        run_cmd("cd Captioner-for-Colab && git pull", "git")
    else:
        run_cmd("git clone https://github.com/GodL-x-SouL/Captioner-for-Colab.git", "git")
        
    # 3. Dependencies
    run_cmd("pip install -r Captioner-for-Colab/requirements.txt", "deps")
    
    # 4. Install Cloudflare
    run_cmd("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb", "cf_install")
    
    # 5. Start server
    steps["server"] = "Running"
    subprocess.Popen("python3 Captioner-for-Colab/captioner.py", shell=True, stdout=open(captioner_log, "w"), stderr=subprocess.STDOUT)
    
    # 6. Start tunnel
    steps["tunnel"] = "Running"
    subprocess.Popen("cloudflared tunnel --url http://127.0.0.1:7860", shell=True, stdout=open(cf_log, "w"), stderr=subprocess.STDOUT)

# Start setup thread
threading.Thread(target=bg_setup, daemon=True).start()

# HTML Builder
def get_html_ui(steps, tunnel_url, logs):
    def get_dot_class(status):
        if status == "Waiting": return "waiting"
        if status == "Running" or status == "Starting": return "running"
        if status == "Completed": return "completed"
        return "failed"
        
    log_content = "\n".join(logs)
    
    tunnel_section = ""
    if tunnel_url:
        tunnel_section = f'''
        <div class="tunnel-box">
            <span style="font-family: \'Cinzel\', serif; font-size: 11px; letter-spacing: 1px; color: #c5a880;">TUNNEL ACTIVE</span>
            <a href="{tunnel_url}" target="_blank" class="tunnel-url">{tunnel_url}</a>
            <span style="font-size: 9px; color: #71717a; margin-top: 10px; text-transform: uppercase; letter-spacing: 1px;">Click to launch studio</span>
        </div>
        '''
    else:
        tunnel_section = '''
        <div class="tunnel-box">
            <span style="font-family: \'Cinzel\', serif; font-size: 11px; letter-spacing: 1px; color: #c5a880; margin-bottom: 8px;">WAITING FOR TUNNEL</span>
            <div class="tunnel-pending">Cloudflare tunnel is initializing...</div>
        </div>
        '''
        
    return f'''
    <div class="studio-container">
        <style>
            @import url(\'https://fonts.googleapis.com/css2?family=Cinzel:wght@400;600&family=Inter:wght@300;400;500&family=JetBrains+Mono&display=swap\');
            .studio-container {{
                width: 95%;
                max-width: 1100px;
                margin: 10px auto;
                background-color: #0c0c0e;
                color: #e4e4e7;
                font-family: \'Inter\', sans-serif;
                border: 1px solid #1a1a20;
                border-radius: 12px;
                padding: 24px;
                box-shadow: 0 10px 40px rgba(0,0,0,0.8);
            }}
            .studio-header {{
                text-align: center;
                border-bottom: 1px solid #1a1a20;
                padding-bottom: 16px;
                margin-bottom: 20px;
            }}
            .studio-title {{ 
                font-family: \'Cinzel\', serif;
                font-size: 24px;
                font-weight: 600;
                letter-spacing: 4px;
                color: #c5a880;
                margin-bottom: 6px;
            }}
            .studio-subtitle {{
                font-size: 10px;
                letter-spacing: 2px;
                text-transform: uppercase;
                color: #71717a;
            }}
            .studio-grid {{
                display: grid;
                grid-template-columns: 1.2fr 1fr;
                gap: 20px;
                margin-bottom: 20px;
            }}
            .studio-card {{
                background: #111115;
                border: 1px solid #1c1c24;
                border-radius: 8px;
                padding: 16px;
            }}
            .card-title {{
                font-family: \'Cinzel\', serif;
                font-size: 12px;
                letter-spacing: 2px;
                color: #c5a880;
                margin-bottom: 12px;
                border-bottom: 1px solid #1c1c24;
                padding-bottom: 6px;
                text-transform: uppercase;
            }}
            .status-item {{
                display: flex;
                align-items: center;
                justify-content: space-between;
                padding: 6px 0;
                border-bottom: 1px solid #15151a;
            }}
            .status-item:last-child {{
                border-bottom: none;
            }}
            .status-label {{
                font-size: 11px;
                font-weight: 400;
                flex: 1;
            }}
            .status-dot-container {{
                display: flex;
                align-items: center;
                gap: 8px;
            }}
            .status-text-val {{
                font-size: 10px;
                color: #71717a;
                font-family: \'JetBrains Mono\', monospace;
            }}
            .status-dot {{
                width: 8px;
                height: 8px;
                border-radius: 50%;
                background-color: #52525b;
            }}
            .status-dot.waiting {{ background-color: #3f3f46; }}
            .status-dot.running {{
                background-color: #d4af37;
                box-shadow: 0 0 8px #d4af37;
                animation: pulse 1.5s infinite;
            }}
            .status-dot.completed {{
                background-color: #10b981;
                box-shadow: 0 0 8px #10b981;
            }}
            .status-dot.failed {{
                background-color: #ef4444;
                box-shadow: 0 0 8px #ef4444;
            }}
            @keyframes pulse {{
                0% {{ transform: scale(0.95); opacity: 0.6; }}
                50% {{ transform: scale(1.05); opacity: 1; }}
                100% {{ transform: scale(0.95); opacity: 0.6; }}
            }}
            .tunnel-box {{
                text-align: center;
                padding: 20px;
                border: 1px dashed #c5a880;
                border-radius: 8px;
                background: #141418;
                display: flex;
                flex-direction: column;
                align-items: center;
                justify-content: center;
                height: calc(100% - 44px);
            }}
            .tunnel-url {{ 
                font-family: \'Inter\', sans-serif;
                font-size: 15px;
                font-weight: 500;
                color: #10b981;
                text-decoration: none;
                border-bottom: 1px solid #10b981;
                padding-bottom: 2px;
                margin-top: 10px;
                transition: all 0.2s;
            }}
            .tunnel-url:hover {{
                color: #34d399;
                border-color: #34d399;
                letter-spacing: 0.5px;
            }}
            .tunnel-pending {{
                font-size: 11px;
                color: #71717a;
                font-style: italic;
            }}
            .logs-container {{ 
                background: #08080a;
                border: 1px solid #181820;
                border-radius: 8px;
                padding: 12px;
            }}
            .logs-box {{
                height: 160px;
                overflow-y: auto;
                font-family: \'JetBrains Mono\', monospace;
                font-size: 10px;
                line-height: 1.5;
                color: #a1a1aa;
                white-space: pre-wrap;
                word-break: break-all;
            }}
            .logs-box::-webkit-scrollbar {{
                width: 4px;
                height: 4px;
            }}
            .logs-box::-webkit-scrollbar-track {{
                background: transparent;
            }}
            .logs-box::-webkit-scrollbar-thumb {{
                background: #27272a;
                border-radius: 2px;
            }}
            .logs-box::-webkit-scrollbar-thumb:hover {{
                background: #c5a880;
            }}
        </style>
        
        <div class="studio-header">
            <div class="studio-title">DATASET CAPTIONER STUDIO</div>
            <div class="studio-subtitle">Google Colab Execution Center</div>
        </div>
        
        <div class="studio-grid">
            <div class="studio-card">
                <div class="card-title">Setup Progress</div>
                
                <div class="status-item">
                    <span class="status-label">Installing Aria2 Downloader</span>
                    <div class='status-dot-container'>
                        <span class="status-text-val">{steps["env"]}</span>
                        <span class="status-dot {get_dot_class(steps["env"])}"></span>
                    </div>
                </div>
                
                <div class="status-item">
                    <span class="status-label">Cloning Captioner Repository</span>
                    <div class='status-dot-container'>
                        <span class="status-text-val">{steps["git"]}</span>
                        <span class="status-dot {get_dot_class(steps["git"])}"></span>
                    </div>
                </div>
                
                <div class="status-item">
                    <span class="status-label">Installing Python Dependencies</span>
                    <div class='status-dot-container'>
                        <span class="status-text-val">{steps["deps"]}</span>
                        <span class="status-dot {get_dot_class(steps["deps"])}"></span>
                    </div>
                </div>
                
                <div class="status-item">
                    <span class="status-label">Installing Cloudflare Client</span>
                    <div class='status-dot-container'>
                        <span class="status-text-val">{steps["cf_install"]}</span>
                        <span class="status-dot {get_dot_class(steps["cf_install"])}"></span>
                    </div>
                </div>
                
                <div class="status-item">
                    <span class="status-label">Starting Captioner Server</span>
                    <div class='status-dot-container'>
                        <span class="status-text-val">{steps["server"]}</span>
                        <span class="status-dot {get_dot_class(steps["server"])}"></span>
                    </div>
                </div>
                
                <div class="status-item">
                    <span class="status-label">Establishing Cloudflare Tunnel</span>
                    <div class='status-dot-container'>
                        <span class="status-text-val">{steps["tunnel"]}</span>
                        <span class="status-dot {get_dot_class(steps["tunnel"])}"></span>
                    </div>
                </div>
            </div>
            
            <div class="studio-card">
                <div class="card-title">Access Endpoint</div>
                {tunnel_section}
            </div>
        </div>
        
        <div class="logs-container">
            <div class="card-title" style="margin-bottom: 8px;">System Logs</div>
            <div class="logs-box" id="notebook-logs">{log_content}</div>
        </div>
    </div>
    '''

# Polling and display loop
try:
    while True:
        # Check Cloudflare logs for tunnel URL
        if steps["tunnel"] in ("Running", "Starting", "Completed"):
            if os.path.exists(cf_log):
                with open(cf_log, "r", encoding="utf-8") as f:
                    content = f.read()
                    match = re.search(r"https://[a-zA-Z0-9-]+\\.trycloudflare\\.com", content)
                    if match:
                        tunnel_url = match.group(0)
                        steps["tunnel"] = "Completed"
                        steps["server"] = "Completed"
                        
        # Read latest lines from the logs to show in console
        current_logs = []
        if steps["server"] in ("Running", "Completed") and os.path.exists(captioner_log):
            with open(captioner_log, "r", encoding="utf-8") as f:
                lines = f.readlines()
                current_logs = [f"[Server] {l.strip()}" for l in lines[-10:]]
        else:
            # show setup logs
            current_logs = log_lines[-10:]
            
        html_code = get_html_ui(steps, tunnel_url, current_logs)
        clear_output(wait=True)
        display(HTML(html_code))
        
        time.sleep(1.0)
except KeyboardInterrupt:
    clear_output(wait=True)
    display(HTML("<div style=\'color: #c5a880; font-family: Inter; padding: 20px; text-align: center; font-size: 14px;\'>Studio Dashboard Polling Stopped. Processes are still running in background.</div>"))
